In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../'))
sys.path.append(base_path)


import tensorflow as tf

2024-03-08 11:02:11.865928: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-03-08 11:02:11.907377: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-08 11:02:11.907421: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-08 11:02:11.908657: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-08 11:02:11.916165: I tensorflow/core/platform/cpu_feature_guar

In [2]:
class Net(tf.keras.Model):
  """A simple linear model."""

  def __init__(self):
    super(Net, self).__init__()
    self.l1 = tf.keras.layers.Dense(5)

  def call(self, x):
    return self.l1(x)


In [3]:
def toy_dataset():
  inputs = tf.range(10.)[:, None]
  labels = inputs * 5. + tf.range(5.)[None, :]
  return tf.data.Dataset.from_tensor_slices(
    dict(x=inputs, y=labels)).repeat().batch(2)

In [4]:
def train_step(net, example, optimizer):
  """Trains `net` on `example` using `optimizer`."""
  with tf.GradientTape() as tape:
    output = net(example['x'])
    loss = tf.reduce_mean(tf.abs(output - example['y']))
  variables = net.trainable_variables
  gradients = tape.gradient(loss, variables)
  optimizer.apply_gradients(zip(gradients, variables))
  return loss

In [5]:
opt = tf.keras.optimizers.Adam(0.1)
dataset = toy_dataset()
iterator = iter(dataset)
ckpt = tf.train.Checkpoint(step=tf.Variable(1), optimizer=opt, net=net, iterator=iterator)
manager = tf.train.CheckpointManager(ckpt, f'{current_path}/models/check_points/', max_to_keep=3)

In [6]:
def train_and_checkpoint(net, manager):
  ckpt.restore(manager.latest_checkpoint)
  if manager.latest_checkpoint:
    print("Restored from {}".format(manager.latest_checkpoint))
  else:
    print("Initializing from scratch.")

  for _ in range(50):
    example = next(iterator)
    loss = train_step(net, example, opt)
    ckpt.step.assign_add(1)
    if int(ckpt.step) % 10 == 0:
      save_path = manager.save()
      print("Saved checkpoint for step {}: {}".format(int(ckpt.step), save_path))
      print("loss {:1.2f}".format(loss.numpy()))

In [7]:
# net = Net()
# train_and_checkpoint(net, manager)

In [8]:
opt = tf.keras.optimizers.Adam(0.1)
net = Net()
dataset = toy_dataset()
iterator = iter(dataset)
ckpt = tf.train.Checkpoint(step=tf.Variable(1), optimizer=opt, net=net, iterator=iterator)
manager = tf.train.CheckpointManager(ckpt, f'{current_path}/models/check_points/', max_to_keep=3)

train_and_checkpoint(net, manager)

2024-03-08 11:02:14.636727: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 864 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB, pci bus id: 0000:c2:00.0, compute capability: 8.0
2024-03-08 11:02:14.841464: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Restored from /cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-10


2024-03-08 11:02:15.131197: I external/local_xla/xla/service/service.cc:168] XLA service 0xb325950 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-03-08 11:02:15.131238: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-PCIE-40GB, Compute Capability 8.0
2024-03-08 11:02:15.137260: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-03-08 11:02:15.159275: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
I0000 00:00:1709892135.226780 2305764 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Saved checkpoint for step 110: /cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-11
loss 0.24
Saved checkpoint for step 120: /cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-12
loss 0.17
Saved checkpoint for step 130: /cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-13
loss 0.19
Saved checkpoint for step 140: /cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-14
loss 0.22
Saved checkpoint for step 150: /cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-15
loss 0.02


In [9]:
print(manager.checkpoints)  # List the three remaining checkpoints

['/cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-13', '/cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-14', '/cluster/home/agustinc/S4/code/test_code/check_points_example/models/check_points/ckpt-15']
